# 3.1.0-Clustering Level 1

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
options(warn = -1, verbose = FALSE)
suppressMessages(suppressWarnings({
library(Signac)
library(Seurat)
library(tidyverse)
library(ggplot2)
library(gridExtra)
}))

## Parameters

In [ ]:
args <- commandArgs(TRUE)

patient <- args[2]
subproj <- args[3]

In [ ]:
source(here::here(glue::glue("{subproj}/multiome_analysis/misc/paths.R")))
source(here::here(glue::glue("{subproj}/multiome_analysis/utils/variables.R")))


"{clust}/{plt_dir}/level_1" %>%
  glue::glue() %>%
  here::here() %>%
  dir.create(path = .,
           showWarnings = FALSE,
           recursive = TRUE)

"{clust}/{robj_dir}/level_1" %>%
  glue::glue() %>%
  here::here() %>%
  dir.create(path = .,
           showWarnings = FALSE,
           recursive = TRUE)

set.seed(0)

In [ ]:
var_name <- paste0("reso_l1_", patient)
reso <- get(var_name)
reso

## Load data

In [ ]:
seurat_obj <- readRDS(here::here(glue::glue("{clust}/{robj_dir}/3.0.0-lognorm_{patient}.rds")))
seurat_obj

In [ ]:
DefaultAssay(seurat_obj) <- "RNA"

## Clustering

In [ ]:
# Define resolutions
resolutions <- seq(from =  0.05, to =  0.7, by =  0.05)
  
# Find clusters
resolutions_dfs <- lapply(resolutions, function(x) {
  seurat_obj <- Seurat::FindClusters(seurat_obj, resolution = x,
                                     graph.name = "wsnn")
  df <- data.frame(
    barcode = colnames(seurat_obj),
    cluster_id = as.character(seurat_obj$seurat_clusters)
  )
  df
})

In [ ]:
# Merge data frames
names(resolutions_dfs) <- resolutions
resolutions_df <- dplyr::bind_rows(resolutions_dfs, .id = "resolution")
resolutions_df$resolution <- as.numeric(resolutions_df$resolution)
head(resolutions_df)

In [ ]:
list_plots <- lapply(unique(resolutions_df$resolution), function(x) {
  p <- add_reso_results(seurat_obj, resolutions_df, resolution=x)
  umap <- DimPlot(p, reduction = "wnn.umap", 
                  group.by=glue::glue("seurat_clusters")) +
    labs(title=glue::glue("Resolution {x}"))
  return(umap)
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol=4)

In [ ]:
immune <- c("PTPRC","CD3E", "CD3D", "CD14", "CD68", "FCGR3A", 
            "CD4", "CD8A", "CD8B", "CD19", "MS4A1", "CD38")

vars2plot <- exp_genes_plot(immune, seurat_obj)

FeaturePlot(seurat_obj, reduction = "wnn.umap", feature=vars2plot, ncol = 4, 
            cols = c("gray", "blue"), min.cutoff = 0, order=T)

In [ ]:
lite_tumor <- c("FABP7", "GLI2", "TYR", "MYC", "MEOX2", "WNT5A", 
                "MEIS1", "MEIS2", "MGP", "BMP4", "BMP5", "MYOC", 
                "SMOC2")

vars2plot <- exp_genes_plot(lite_tumor, seurat_obj)

FeaturePlot(seurat_obj, reduction = "wnn.umap", feature=vars2plot, ncol = 4, 
            cols = c("gray", "blue"),  min.cutoff = 0)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
qc_vars <- c('nCount_ATAC','nFeature_ATAC','nCount_RNA','nFeature_RNA',
             "doublet_score","doublet_score_atac",
             "doublet_enrichment_atac", "pct_mt")


FeaturePlot(seurat_obj, reduction = "wnn.umap", feature=qc_vars, ncol = 4, 
            cols = c("gray", "blue"), min.cutoff = 0, order=T)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
other <- c("SMARCB1", "PECAM1","ICAM1","STMN1","MKI67","TOP2A")


FeaturePlot(seurat_obj, reduction = "wnn.umap", feature=other, ncol = 4, 
            cols = c("gray", "blue"), min.cutoff = 0, order=T)

In [ ]:
seurat_obj <- FindClusters(seurat_obj, resolution = as.numeric(reso),
                                     graph.name = "wsnn")

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
DimPlot(seurat_obj, reduction = "wnn.umap",  group.by=glue::glue("seurat_clusters"))

In [ ]:
column1 <- paste0("clusters_reso_", reso)
seurat_obj[[column1]]<- Idents(seurat_obj)

In [ ]:
rna.markers <- FindAllMarkers(seurat_obj,only.pos = TRUE, min.pct = 0.1, logfc.threshold = 0.2)

## Save

In [ ]:
saveRDS(seurat_obj, here::here(glue::glue("{clust}/{robj_dir}/level_1/3.1.0-l1_anno_{patient}.rds")))

In [ ]:
readr::write_csv(rna.markers, here::here(glue::glue("{clust}/{robj_dir}/level_1/3.1.0-l1_anno_{patient}.csv")))

## Session

In [ ]:
sessionInfo()